# 11. Broadcasting

Broadcasting is how NumPy does arithmetic on arrays of **different shapes** without copying data. You've already used it every time you wrote `arr + 1`. Once you understand the rules, a lot of loops just disappear.

In [1]:
import numpy as np

## The rules

NumPy compares the shapes **from the right**, one dimension at a time. Two dimensions are compatible if:

1. they're equal, or
2. one of them is 1.

If one array has fewer dimensions, it's treated as if it had extra 1s on the **left**. Dimensions of size 1 get stretched to match the other array.

| A | B | Result |
|---|---|---|
| `(3, 4)` | `()` (scalar) | `(3, 4)` |
| `(3, 4)` | `(4,)` | `(3, 4)` |
| `(3, 4)` | `(3, 1)` | `(3, 4)` |
| `(3, 1)` | `(1, 4)` | `(3, 4)` |
| `(8, 1, 6)` | `(7, 1)` | `(8, 7, 6)` |
| `(3, 4)` | `(3,)` | **error**: 4 vs 3 |

## Scalar with array

In [2]:
a = np.arange(12).reshape(3, 4)
a * 10

array([[  0,  10,  20,  30],
       [ 40,  50,  60,  70],
       [ 80,  90, 100, 110]])

## Row vector with matrix

A shape `(4,)` array lines up with the **last** axis, so it gets added to every row:

In [3]:
row = np.array([100, 200, 300, 400])
a + row

array([[100, 201, 302, 403],
       [104, 205, 306, 407],
       [108, 209, 310, 411]])

## Column vector with matrix

To add something to every column instead, the vector needs shape `(3, 1)`. A plain `(3,)` array won't work, because it gets compared against the 4:

In [4]:
col = np.array([1000, 2000, 3000])
try:
    a + col
except ValueError as e:
    print('ValueError:', e)

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


In [5]:
a + col[:, np.newaxis]     # or col.reshape(-1, 1)

array([[1000, 1001, 1002, 1003],
       [2004, 2005, 2006, 2007],
       [3008, 3009, 3010, 3011]])

## Two vectors make a grid

`(3, 1)` with `(1, 4)` gives `(3, 4)`. This is a neat way to build tables without loops, like a multiplication table:

In [6]:
x = np.arange(1, 10)
x[:, None] * x[None, :]

array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9],
       [ 2,  4,  6,  8, 10, 12, 14, 16, 18],
       [ 3,  6,  9, 12, 15, 18, 21, 24, 27],
       [ 4,  8, 12, 16, 20, 24, 28, 32, 36],
       [ 5, 10, 15, 20, 25, 30, 35, 40, 45],
       [ 6, 12, 18, 24, 30, 36, 42, 48, 54],
       [ 7, 14, 21, 28, 35, 42, 49, 56, 63],
       [ 8, 16, 24, 32, 40, 48, 56, 64, 72],
       [ 9, 18, 27, 36, 45, 54, 63, 72, 81]])

`np.broadcast_shapes` tells you what shape you'll get, without doing the computation. Useful for checking your reasoning:

In [7]:
print(np.broadcast_shapes((8, 1, 6), (7, 1)))
print(np.broadcast_shapes((5, 3), (3,)))

(8, 7, 6)
(5, 3)


## Practical examples

### Standardising columns

Subtract each column's mean and divide by its std. `mean(axis=0)` has shape `(n_features,)`, which broadcasts across the rows:

In [8]:
rng = np.random.default_rng(0)
X = rng.normal(loc=[10, 50, -3], scale=[1, 20, 0.5], size=(500, 3))

Z = (X - X.mean(axis=0)) / X.std(axis=0)
print(Z.mean(axis=0).round(6) + 0.0)   # + 0.0 just tidies up -0.0
print(Z.std(axis=0).round(6))

[0. 0. 0.]
[1. 1. 1.]


### Normalising rows

For row-wise operations, use `keepdims=True` so the result stays as a column:

In [9]:
counts = np.array([[1, 3, 6], [2, 2, 0], [5, 0, 5]])
probs = counts / counts.sum(axis=1, keepdims=True)
probs

array([[0.1, 0.3, 0.6],
       [0.5, 0.5, 0. ],
       [0.5, 0. , 0.5]])

### Pairwise distances

Given `n` points and `m` centres, compute every point-to-centre distance at once. Shapes: `(n, 1, 2) - (1, m, 2)` gives `(n, m, 2)`.

In [10]:
points = rng.random((5, 2))
centres = np.array([[0.1, 0.1], [0.9, 0.9], [0.1, 0.9]])

diff = points[:, None, :] - centres[None, :, :]
dist = np.sqrt((diff ** 2).sum(axis=-1))
print(dist.shape)
print(dist.round(3))
print('closest centre for each point:', dist.argmin(axis=1))

(5, 3)
[[0.576 0.722 0.264]
 [0.317 0.854 0.515]
 [0.269 0.87  0.691]
 [0.705 0.609 0.901]
 [0.819 0.711 1.081]]
closest centre for each point: [2 0 0 1 1]


### Applying a colour tint to an image

An RGB image has shape `(H, W, 3)`. Multiplying by a length-3 array scales each channel:

In [11]:
img = np.ones((2, 2, 3))
tint = np.array([1.0, 0.5, 0.2])
(img * tint)[0, 0]

array([1. , 0.5, 0.2])

## Things to watch out for

**Accidental broadcasting.** A shape mismatch that *happens* to be compatible gives you a big matrix instead of an error. This one catches people all the time:

In [12]:
y_true = np.array([1., 2., 3.])
y_pred = np.array([[1.1], [1.9], [3.2]])   # (3, 1), e.g. straight out of a model

err = y_true - y_pred
print(err.shape)    # (3, 3), not (3,)!
print('wrong MSE:', (err ** 2).mean())
print('right MSE:', ((y_true - y_pred.ravel()) ** 2).mean())

(3, 3)
wrong MSE: 1.4200000000000004
right MSE: 0.020000000000000035


**Memory.** Broadcasting itself doesn't copy, but the *result* has the full broadcast shape. The pairwise-distance trick on 100k × 100k points would need 80 GB for the result alone.

If you want a broadcast array without computing anything (as a read-only view), use `np.broadcast_to`:

In [13]:
b = np.broadcast_to(np.arange(3), (4, 3))
print(b)
print(b.strides)    # stride 0 on the first axis: the same row is reused

[[0 1 2]
 [0 1 2]
 [0 1 2]
 [0 1 2]]
(0, 8)
